In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import joblib
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

I0000 00:00:1779907759.386866  185196 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
df = pd.read_csv("Data_Clean.csv")
df['tanggal_nabung'] = pd.to_datetime(df['tanggal_nabung'])
df = df.sort_values(by=['id_tabungan', 'tanggal_nabung'])

selesai_ids = df[df['status'] == 'Selesai']['id_tabungan'].unique()
df_train = df[df['id_tabungan'].isin(selesai_ids)].copy()

total_langkah = df_train.groupby('id_tabungan')['counter_tabungan'].transform('max')
df_train['sisa_kali_nabung'] = total_langkah - df_train['counter_tabungan']

df_train['sisa_nominal'] = df_train['target_nominal'] - df_train['total_terkumpul']
df_train['rumus_kalkulator'] = np.ceil(df_train['sisa_nominal'] / (df_train['nominal_nabung'] + 1)) 

fitur_x = [
    'target_nominal', 'nominal_nabung', 'total_terkumpul', 
    'jarak_hari_nabung', 'sisa_nominal', 'rumus_kalkulator'
]

print(f"Total baris data untuk training: {len(df_train)}")
print(f"Jumlah fitur sekarang: {len(fitur_x)}")

Total baris data untuk training: 24202
Jumlah fitur sekarang: 6


In [3]:
scaler_x = StandardScaler()
df_train[fitur_x] = scaler_x.fit_transform(df_train[fitur_x])

scaler_filename = "scaler_lstm.pkl"
joblib.dump(scaler_x, scaler_filename)
print(f"💾 Scaler X berhasil di-fit dan disimpan sebagai: {scaler_filename}")

💾 Scaler X berhasil di-fit dan disimpan sebagai: scaler_lstm.pkl


In [4]:
SEQ_LENGTH = 5

def create_sequences(data, seq_length, feature_cols, target_col):
    xs, ys = [], []
    for id_tab, group in data.groupby('id_tabungan'):
        group_features = group[feature_cols].values
        group_target = group[target_col].values
        
        for i in range(len(group)):
            if i < seq_length - 1:
                pad_size = seq_length - 1 - i
                pad = np.zeros((pad_size, len(feature_cols)))
                seq = np.vstack([pad, group_features[:i+1]])
            else:
                seq = group_features[i - seq_length + 1 : i + 1]
                
            xs.append(seq)
            ys.append(group_target[i])
            
    return np.array(xs), np.array(ys)

X, y_raw = create_sequences(df_train, SEQ_LENGTH, fitur_x, 'sisa_kali_nabung')

scaler_y = StandardScaler()
y_scaled = scaler_y.fit_transform(y_raw.reshape(-1, 1)).flatten()

joblib.dump(scaler_y, "scaler_y_lstm.pkl")
print("💾 Scaler Y berhasil di-fit dan disimpan sebagai: scaler_y_lstm.pkl")

X_train, X_test, y_train, y_test = train_test_split(X, y_scaled, test_size=0.2, random_state=42)

print(f"Bentuk Input X_train : {X_train.shape} -> (Sampel, Time_Steps, Fitur)")
print(f"Bentuk Target y_train: {y_train.shape}")

💾 Scaler Y berhasil di-fit dan disimpan sebagai: scaler_y_lstm.pkl
Bentuk Input X_train : (19361, 5, 6) -> (Sampel, Time_Steps, Fitur)
Bentuk Target y_train: (19361,)


In [5]:
inputs = layers.Input(shape=(SEQ_LENGTH, len(fitur_x)), name="lstm_input")

x = layers.LSTM(128, return_sequences=True)(inputs)
x = layers.Dropout(0.2)(x)
x = layers.LSTM(64, return_sequences=False)(x)
x = layers.Dense(32, activation='relu')(x)

outputs = layers.Dense(1, activation='linear', name="estimasi_output")(x)

model_lstm = Model(inputs=inputs, outputs=outputs, name="LSTM_Behavioral_Fixed")

model_lstm.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001), 
    loss='mse', 
    metrics=['mae']
)

model_lstm.summary()

W0000 00:00:1779907763.478502  185196 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "LSTM_Behavioral_Fixed"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_input (InputLayer)         │ (None, 5, 6)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 5, 128)         │        69,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 5, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ estimasi_output (Dense)         │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 120,641 (471.25 KB)

 Trainable params: 120,641 (471.25 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
class TargetTercapaiCallback(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs.get('mae') is not None and logs.get('mae') < 0.05:
            print(f"\n[Custom Callback] Epoch {epoch+1}: MAE mencapai target (< 0.05)! Menghentikan proses training.")
            self.model.stop_training = True

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001)

custom_early_stop = TargetTercapaiCallback()

print("Memulai Training LSTM...")
history = model_lstm.fit(
    X_train, y_train,
    epochs=100, 
    batch_size=32, 
    validation_data=(X_test, y_test),
    callbacks=[early_stop, reduce_lr, custom_early_stop],
    verbose=1
)

Memulai Training LSTM...
Epoch 1/100
606/606 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.2491 - mae: 0.2864 - val_loss: 0.1028 - val_mae: 0.1888 - learning_rate: 0.0010
Epoch 2/100
606/606 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0720 - mae: 0.1529 - val_loss: 0.0401 - val_mae: 0.1188 - learning_rate: 0.0010
Epoch 3/100
606/606 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0349 - mae: 0.1138 - val_loss: 0.0282 - val_mae: 0.1135 - learning_rate: 0.0010
Epoch 4/100
606/606 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0261 - mae: 0.1016 - val_loss: 0.0249 - val_mae: 0.1098 - learning_rate: 0.0010
Epoch 5/100
606/606 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0230 - mae: 0.0974 - val_loss: 0.0177 - val_mae: 0.0879 - learning_rate: 0.0010
Epoch 6/100
606/606 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0209 - mae: 0.0936 - val_loss: 0.0173 - val_mae: 0.0860 - learning_rate: 0.0010
Epoch 7/100
606/606 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 0.0196 - mae: 0.0908 - val_loss: 0.0181 - val_mae: 0.089

In [7]:
print("=" * 50)
print("  EVALUASI MODEL (DEEP LEARNING - LSTM)  ")
print("=" * 50)

scaler_y_loaded = joblib.load("scaler_y_lstm.pkl")

dl_preds_scaled = model_lstm.predict(X_test, verbose=0)

dl_preds_raw = scaler_y_loaded.inverse_transform(dl_preds_scaled)
dl_preds = np.round(dl_preds_raw.flatten())

y_test_asli = scaler_y_loaded.inverse_transform(y_test.reshape(-1, 1)).flatten()

def evaluate(name, true, pred):
    print(f"\n--- {name} ---")
    print(f"MAE  : {mean_absolute_error(true, pred):.4f} kali nabung")
    print(f"R2   : {r2_score(true, pred):.4f}")

evaluate("Deep Learning (LSTM)", y_test_asli, dl_preds)

  EVALUASI MODEL (DEEP LEARNING - LSTM)  

--- Deep Learning (LSTM) ---
MAE  : 1.1774 kali nabung
R2   : 0.9882


In [8]:
model_filename = "model_lstm_tabungan.keras"
model_lstm.save(model_filename)
print(f"💾 Model berhasil disimpan sebagai: {model_filename}")

💾 Model berhasil disimpan sebagai: model_lstm_tabungan.keras


In [9]:
print("\n" + "="*70)
print("  UJI COBA PREDIKSI PRODUKSI (INFERENCE LSTM)")
print("="*70)

loaded_model = keras.models.load_model(model_filename)
loaded_scaler_x = joblib.load("scaler_lstm.pkl")
loaded_scaler_y = joblib.load("scaler_y_lstm.pkl")

data_histori = pd.DataFrame([
    [5000000, 500000, 500000, 0],   
    [5000000, 600000, 1100000, 7],  
    [5000000, 200000, 1300000, 30]  
], columns=['target_nominal', 'nominal_nabung', 'total_terkumpul', 'jarak_hari_nabung'])

data_histori['sisa_nominal'] = data_histori['target_nominal'] - data_histori['total_terkumpul']
data_histori['rumus_kalkulator'] = np.ceil(data_histori['sisa_nominal'] / (data_histori['nominal_nabung'] + 1))

riwayat_scaled = loaded_scaler_x.transform(data_histori)

jumlah_padding = SEQ_LENGTH - len(riwayat_scaled)
padding = np.zeros((jumlah_padding, len(data_histori.columns)))
riwayat_final = np.vstack([padding, riwayat_scaled]) 

input_tensor = np.expand_dims(riwayat_final, axis=0)

prediksi_scaled = loaded_model.predict(input_tensor, verbose=0)
prediksi_raw = loaded_scaler_y.inverse_transform(prediksi_scaled) 
estimasi_kali_nabung = np.ceil(prediksi_raw[0][0])

print(f"Riwayat transaksi diproses: {len(data_histori)} transaksi")
print(f"Prediksi AI (Behavioral)  : Sekitar {estimasi_kali_nabung} kali nabung lagi")


  UJI COBA PREDIKSI PRODUKSI (INFERENCE LSTM)
Riwayat transaksi diproses: 3 transaksi
Prediksi AI (Behavioral)  : Sekitar 10.0 kali nabung lagi
